# v4 — `geo_cluster`: spatial neighbourhoods from KMeans on lat/long

**What v4 adds.** One feature: `geo_cluster`, a categorical from `KMeans(n_clusters=30)` on `[latitude, longitude]`. Everything else is v3's pipeline, search and early stopping, **including balanced class weights**.

**Hypotheses.**
- **H7a** — a spatial-neighbourhood feature adds accuracy: nearby pumps share infrastructure and upkeep, and raw lat/long splits express that clumsily.
- **H5″** — carried over from v2/v3: weights hurt accuracy. This notebook runs **without weights** as the main protocol, since the v3 notebook found the honest best was ≈ 0.811 only without them.
- **H1 (retest)** — the latitude-placeholder bug (`-2e-8`) feeds directly into the cluster step, so the fix may matter *here* even though it did not at v2.
- v4's own doc warns the cluster may merely rediscover `region`. We check that.

**Protocol.** Searched params from the v3 notebook (`cache/v3_search.json`), fixed 300 trees (early stopping added nothing at v3), **no weights**, same folds. v4 re-ran its own search *with* the cluster feature; we reuse v3's params so that only the feature changes.

| row | change vs reference | weights |
|---|---|---|
| ref | v3 row 3a: no cluster (read from `results.jsonl`) | no |
| 4a | + `geo_cluster`, k = 30 (**v4's feature**) | no |
| 4b | k = 10 | no |
| 4c | k = 60 | no |
| 4d | k = 30 + latitude fix | no |
| 4e | latitude fix only, no cluster | no |
| 4f | **v4 as shipped**: k = 30, early stopping on val fold, **weights** | yes |

In [1]:
import sys, warnings, json
sys.path.insert(0, '.')
warnings.simplefilter('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import *
pd.set_option('display.width', 140)
df, y, le = get_train('v4')
Flex = make_flex('v4')
BEST = json.loads(open('cache/v3_search.json').read())['best_params']
ref = next(r for r in map(json.loads, RESULTS.read_text().splitlines()) if r['label'].startswith('3a '))
print('params:', BEST); print(f"reference 3a (v3, no cluster, no weights): {ref['mean']:.4f} ± {ref['std']:.4f}")

def run(label, flex_kw=None, **kw):
    sc, oof, it = cv_xgb_es(df, y, lambda: Flex(**(flex_kw or {})), BEST, **kw)
    r = log_result('v4', label, sc)
    d = np.array(sc) - np.array(ref['folds'])
    print(f"{label:<52} {r['mean']:.4f} ± {r['std']:.4f}   Δ vs ref {d.mean():+.4f}  per-fold {np.round(d, 4).tolist()}")
    return sc, oof

params: {'subsample': 0.6, 'min_child_weight': 5, 'max_depth': 12, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
reference 3a (v3, no cluster, no weights): 0.8110 ± 0.0038


## Is the cluster just `region` again? (redundancy check)

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_mutual_info_score as ami
F = Flex(k=30); Xf = F.fit_transform(df.drop(columns=[TARGET]))
print('AMI(geo_cluster, region)       :', round(ami(Xf.geo_cluster, Xf.region), 3), '  (1 = identical partition)')
print('AMI(geo_cluster, target)       :', round(ami(Xf.geo_cluster, df[TARGET]), 3))
print('AMI(region, target)            :', round(ami(Xf.region, df[TARGET]), 3))
print('AMI(lga, target)               :', round(ami(Xf.lga, df[TARGET]), 3))
print('cluster sizes: min %d, median %d, max %d' % tuple(Xf.geo_cluster.value_counts().agg(['min', 'median', 'max'])))

AMI(geo_cluster, region)       : 0.804   (1 = identical partition)
AMI(geo_cluster, target)       : 0.021


AMI(region, target)            : 0.021
AMI(lga, target)               : 0.033
cluster sizes: min 510, median 1883, max 3628


## 4a — `geo_cluster`, k = 30, no weights

In [3]:
sc_a, oof_a = run('4a + geo_cluster k=30, no weights', dict(k=30), es='none')

4a + geo_cluster k=30, no weights                    0.8111 ± 0.0034   Δ vs ref +0.0001  per-fold [-0.0007, 0.0011, 0.0015, 0.0001, -0.0017]


## 4b / 4c — cluster-count sweep (10, 60)

In [4]:
sc_b, _ = run('4b k=10, no weights', dict(k=10), es='none')
sc_c, _ = run('4c k=60, no weights', dict(k=60), es='none')

4b k=10, no weights                                  0.8119 ± 0.0040   Δ vs ref +0.0008  per-fold [0.0007, 0.0013, 0.0013, -0.0001, 0.001]


4c k=60, no weights                                  0.8118 ± 0.0034   Δ vs ref +0.0007  per-fold [0.0002, 0.001, 0.001, 0.0013, 0.0002]


## 4d / 4e — latitude fix, with and without the cluster

In [5]:
sc_d, _ = run('4d k=30 + latitude fix, no weights', dict(k=30, fix_lat=True), es='none')
sc_e, _ = run('4e latitude fix only (no cluster), no weights', dict(fix_lat=True), es='none')

4d k=30 + latitude fix, no weights                   0.8112 ± 0.0049   Δ vs ref +0.0002  per-fold [0.0025, 0.0008, 0.0007, -0.001, -0.0021]


4e latitude fix only (no cluster), no weights        0.8106 ± 0.0036   Δ vs ref -0.0004  per-fold [-0.0003, -0.0001, 0.0002, 0.0, -0.0017]


## 4f — v4 as shipped (k = 30, early stopping on val fold, balanced weights)

In [6]:
sc_f, oof_f = run('4f v4 as shipped (val ES + weights)', dict(k=30), es='val', balanced=True)

4f v4 as shipped (val ES + weights)                  0.7813 ± 0.0031   Δ vs ref -0.0297  per-fold [-0.0298, -0.0306, -0.0282, -0.0276, -0.0322]


## Does the model actually use it?

In [7]:
import xgboost as xgb
clf = xgb.XGBClassifier(**BEST, n_estimators=300, tree_method='hist', enable_categorical=True,
                        random_state=SEED, n_jobs=-1).fit(Xf, y)
imp = pd.Series(clf.get_booster().get_score(importance_type='total_gain')).sort_values(ascending=False)
imp = (imp / imp.sum()).round(4)
print('geo_cluster rank by total gain:', list(imp.index).index('geo_cluster') + 1, 'of', len(imp))
imp.head(10)

geo_cluster rank by total gain: 16 of 42


ward                     0.1617
quantity                 0.1427
lga                      0.1349
waterpoint_type          0.0653
funder                   0.0564
installer                0.0497
extraction_type_class    0.0375
population               0.0328
gps_height               0.0296
latitude                 0.0271
dtype: float64

## Results

In [8]:
res = pd.DataFrame([r for r in map(json.loads, RESULTS.read_text().splitlines()) if r['version'] == 'v4'])
res['acc'] = res['mean'].round(4); res['fold std'] = res['std'].round(4)
res['Δ vs ref'] = (res['mean'] - ref['mean']).round(4)
res[['label', 'acc', 'fold std', 'Δ vs ref']].set_index('label')

,acc,fold std,Δ vs ref
label,,,
"4a + geo_cluster k=30, no weights",0.8111,0.0034,0.0001
"4b k=10, no weights",0.8119,0.0040,0.0008
"4c k=60, no weights",0.8118,0.0034,0.0007
"4d k=30 + latitude fix, no weights",0.8112,0.0049,0.0002
"4e latitude fix only (no cluster), no weights",0.8106,0.0036,-0.0004
4f v4 as shipped (val ES + weights),0.7813,0.0031,-0.0297


In [9]:
from sklearn.metrics import recall_score
rows = [{'model': n, 'acc': (o == y).mean(), **{f'recall[{c}]': v for c, v in zip(le.classes_, recall_score(y, o, average=None))}}
        for n, o in [('4a cluster, no weights', oof_a), ('4f v4 shipped (weights)', oof_f)]]
pd.DataFrame(rows).set_index('model').round(4)

,acc,recall[functional],recall[functional needs repair],recall[non functional]
model,,,,
"4a cluster, no weights",0.8111,0.9031,0.3081,0.7761
4f v4 shipped (weights),0.7813,0.8014,0.5851,0.7901


## Conclusion

All rows: searched params, 300 trees, same folds. Reference = v3 row 3a (no cluster, no weights) **0.8110 ± 0.0038**.

| row | CV accuracy | Δ vs ref |
|---|---|---|
| 4a `geo_cluster` k=30 (v4's feature) | 0.8111 ± 0.0034 | +0.0001 |
| 4b k=10 | 0.8119 ± 0.0040 | +0.0008 |
| 4c k=60 | 0.8118 ± 0.0034 | +0.0007 |
| 4d k=30 + latitude fix | 0.8112 ± 0.0049 | +0.0002 |
| 4e latitude fix only | 0.8106 ± 0.0036 | −0.0004 |
| **4f v4 as shipped (val ES + weights)** | **0.7813 ± 0.0031** | **−0.0297** |

- **H7a (geo_cluster adds accuracy) is not supported.** At k=30 the gain is +0.0001 with mixed per-fold signs. k=10 and k=60 reach +0.0007 to +0.0008, which is about a fifth of the fold std (~0.004), so I treat them as noise even where the sign is consistent (k=60 is positive on all 5 folds). The cluster count doesn't rescue the feature.
- **Why: it's `region` again.** AMI(geo_cluster, region) = **0.80**, so the 30 clusters mostly reproduce the region partition. Their relation to the target is the same (AMI 0.021 for both). The model also ranks `geo_cluster` only **16th of 42** by total gain, while raw `latitude` and `longitude` still carry the geographic signal. The warning in v4's own doc was right.
- **H1 (latitude fix) is not supported, again.** Alone −0.0004, with the cluster +0.0001 over 4a. The bug is real but harmless for tree models, even where a clustering step takes the broken coordinates directly.
- **H5″ confirmed a third time.** v4 as shipped, with weights, scores 0.7813, essentially identical to v3 as shipped (0.7818, difference −0.0005, mixed per-fold). Weights cost about −0.03 whichever feature set is in use. v4's feature changes the shipped score by nothing measurable.
- **Side finding.** The highest-gain features are the high-cardinality categoricals `ward` (16% of total gain) and `lga` (13%), ahead of `quantity` (14%). Gain-importance favours high-cardinality splits, so this may be real signal or overfitting to category identity. Worth testing by cutting these two or collapsing rare levels, which is v5's `funder`/`installer` idea.

**Caveats.** One seed and one fold scheme. The params were searched on v3's features and reused (v4 re-searched), so the cluster feature gets no tuning of its own. CV only, no leaderboard.

**Running total.** Honest best is still ≈ 0.811 (v3 row 3c, or any no-weights row here). v4 adds nothing. Next: **v5** cleans `funder`/`installer`, directly relevant to the high-cardinality finding above.